# 04 — Statistical Analysis
**Data sources:**
- `data/processed/master_listings.parquet`
- `data/processed/calendar_singapore_enriched.parquet`
- `data/processed/calendar_bangkok_enriched.parquet`

**Hypotheses tested:**
- **H1:** Entire-home listings command a significantly higher price than private rooms (Mann-Whitney U)
- **H4:** Neighbourhood price differences are statistically significant (one-way ANOVA)
- **H5:** Weekend listings have significantly lower availability than weekdays — tested using enriched calendar data (listing base price inner-joined onto each calendar day)

**Other analyses:**
- Correlation matrix: what numeric factors drive price?
- OLS regression: price ~ room_type + availability + reviews + minimum_nights
- Cross-city price comparison in USD
- Cross-city comparison: are the same drivers significant in both markets?

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from scipy import stats
import statsmodels.formula.api as smf
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

from pathlib import Path
DATA = Path('../data/processed')
master = pd.read_parquet(DATA / 'master_listings.parquet')
sg = master[master['city'] == 'singapore'].copy()
bk = master[master['city'] == 'bangkok'].copy()

print(f'Master: {len(master):,} rows | Singapore: {len(sg):,} | Bangkok: {len(bk):,}')

---
## H1 — Entire Home vs Private Room Price Difference

**Why Mann-Whitney U instead of t-test:**  
Price is highly skewed (Singapore skewness 11.3, Bangkok 53.2). The t-test assumes normality — violated here. Mann-Whitney U is non-parametric: it tests whether one group's values tend to be higher than the other's, with no distributional assumption.

**Null hypothesis (H0):** The price distributions of entire-home and private-room listings are identical.  
**Alternative (H1):** Entire-home listings have significantly higher prices.

In [ ]:
def mann_whitney_room_type(df, city, currency):
    entire = df[df['room_type'] == 'Entire home/apt']['price'].dropna()
    private = df[df['room_type'] == 'Private room']['price'].dropna()

    stat, p = stats.mannwhitneyu(entire, private, alternative='greater')

    # Effect size: rank-biserial correlation
    n1, n2 = len(entire), len(private)
    r = 1 - (2 * stat) / (n1 * n2)

    print(f'\n=== {city} ===')
    print(f'  Entire home  — n={n1:,}, median={currency}{entire.median():,.0f}')
    print(f'  Private room — n={n2:,}, median={currency}{private.median():,.0f}')
    print(f'  Price premium: {entire.median()/private.median():.2f}x')
    print(f'  Mann-Whitney U: {stat:,.0f}')
    print(f'  p-value: {p:.2e}  → {"REJECT H0 (significant)" if p < 0.05 else "FAIL TO REJECT H0"}')
    print(f'  Effect size (rank-biserial r): {r:.3f}  ({"large" if abs(r) > 0.3 else "medium" if abs(r) > 0.1 else "small"})')
    return {'city': city, 'median_entire': entire.median(), 'median_private': private.median(),
            'premium': entire.median()/private.median(), 'p_value': p, 'effect_r': r}

results = []
results.append(mann_whitney_room_type(sg, 'Singapore', 'SGD '))
results.append(mann_whitney_room_type(bk, 'Bangkok', 'THB '))

print('\n=== CROSS-CITY COMPARISON ===')
print(pd.DataFrame(results)[['city','premium','p_value','effect_r']].to_string(index=False))

In [ ]:
# Visualise: violin plot showing full distributions + median markers
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, (df, city, currency, cap) in zip(axes, [
    (sg, 'Singapore', 'SGD', sg['price'].quantile(0.99)),
    (bk, 'Bangkok',   'THB', bk['price'].quantile(0.99)),
]):
    plot = df[df['room_type'].isin(['Entire home/apt', 'Private room']) & (df['price'] <= cap)]
    sns.violinplot(data=plot, x='room_type', y='price', palette='Blues_d', ax=ax, cut=0)
    medians = plot.groupby('room_type')['price'].median()
    for i, rt in enumerate(['Entire home/apt', 'Private room']):
        if rt in medians:
            ax.text(i, medians[rt], f' {currency}{medians[rt]:,.0f}', va='center', fontsize=9, color='white', fontweight='bold')
    ax.set_title(f'{city} — Price by Room Type\n(capped at 99th pct)', fontsize=11)
    ax.set_xlabel('')
    ax.set_ylabel(f'Price ({currency}/night)')
    ax.yaxis.set_major_formatter(mtick.StrMethodFormatter('{x:,.0f}'))

plt.suptitle('H1: Entire Home vs Private Room — Price Distribution', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

---
## H4 — Neighbourhood Price Differences (ANOVA)

**Why ANOVA:** We have more than 2 groups (neighbourhoods). Running multiple t-tests inflates the false positive rate. One-way ANOVA tests whether *any* neighbourhood has a significantly different mean price.

**Null hypothesis (H0):** All neighbourhoods have the same mean price.  
**Alternative (H4):** At least one neighbourhood has a significantly different price.

Note: Bangkok has no `neighbourhood_group` — we run ANOVA across individual neighbourhoods directly.

In [ ]:
def anova_neighbourhood(df, city, col='neighbourhood', min_n=10):
    # Keep only neighbourhoods with enough listings for stable estimates
    counts = df[col].value_counts()
    valid  = counts[counts >= min_n].index
    df_filt = df[df[col].isin(valid)]

    groups = [grp['price'].dropna().values for _, grp in df_filt.groupby(col)]
    f_stat, p = stats.f_oneway(*groups)

    print(f'\n=== {city} — ANOVA on {col} ===')
    print(f'  Groups tested : {len(groups)}')
    print(f'  F-statistic   : {f_stat:.2f}')
    print(f'  p-value       : {p:.2e}  → {"REJECT H0 (price differs by neighbourhood)" if p < 0.05 else "FAIL TO REJECT H0"}')

    summary = (
        df_filt.groupby(col)['price']
        .agg(['median', 'mean', 'count'])
        .sort_values('median', ascending=False)
    )
    print(f'\n  Top 5 by median price:')
    print(summary.head(5).to_string())
    print(f'\n  Bottom 5 by median price:')
    print(summary.tail(5).to_string())

anova_neighbourhood(sg, 'Singapore', 'neighbourhood_group', min_n=5)
anova_neighbourhood(sg, 'Singapore', 'neighbourhood', min_n=10)
anova_neighbourhood(bk, 'Bangkok',   'neighbourhood', min_n=10)

In [ ]:
# Visualise: top 10 vs bottom 10 neighbourhood median prices, both cities
fig, axes = plt.subplots(2, 2, figsize=(15, 11))

for row, (df, city, currency) in enumerate([(sg, 'Singapore', 'SGD'), (bk, 'Bangkok', 'THB')]):
    nb = (
        df.groupby('neighbourhood')['price']
        .agg(['median', 'count'])
        .query('count >= 10')
        .sort_values('median')
        .rename(columns={'median': 'median_price'})
    )
    top10 = nb.tail(10)
    bot10 = nb.head(10)

    axes[row][0].barh(top10.index, top10['median_price'],
                      color=sns.color_palette('Blues_d', 10), edgecolor='white')
    axes[row][0].set_title(f'{city} — Top 10 Neighbourhoods by Median Price', fontsize=10)
    axes[row][0].set_xlabel(f'Median Price ({currency}/night)')

    axes[row][1].barh(bot10.index, bot10['median_price'],
                      color=sns.color_palette('Oranges_d', 10), edgecolor='white')
    axes[row][1].set_title(f'{city} — Bottom 10 Neighbourhoods by Median Price', fontsize=10)
    axes[row][1].set_xlabel(f'Median Price ({currency}/night)')

plt.suptitle('H4: Neighbourhood Price Differences', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

---
## Correlation Matrix — What Drives Price?

Pearson correlation between price and all numeric features, per city.  
**Interpretation:** Values close to +1 or -1 indicate a strong linear relationship with price.  
**Caveat:** Correlation ≠ causation. High correlation with `minimum_nights` might reflect luxury listing policies, not minimum_nights *causing* higher prices.

In [ ]:
NUMERIC_COLS = [
    'price', 'availability_365', 'minimum_nights',
    'number_of_reviews', 'reviews_per_month',
    'calculated_host_listings_count', 'occupancy_proxy', 'review_count'
]

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

for ax, (df, city) in zip(axes, [(sg, 'Singapore'), (bk, 'Bangkok')]):
    cols  = [c for c in NUMERIC_COLS if c in df.columns]
    corr  = df[cols].corr()
    mask  = np.triu(np.ones_like(corr, dtype=bool))
    sns.heatmap(
        corr, mask=mask, ax=ax, annot=True, fmt='.2f',
        cmap='RdBu_r', center=0, vmin=-1, vmax=1,
        square=True, linewidths=0.5, cbar_kws={'shrink': 0.8}
    )
    ax.set_title(f'{city} — Correlation Matrix', fontsize=12)
    ax.tick_params(axis='x', rotation=45)

plt.suptitle('Numeric Feature Correlations (lower triangle)', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

# Print price correlations ranked
for df, city in [(sg, 'Singapore'), (bk, 'Bangkok')]:
    cols = [c for c in NUMERIC_COLS if c in df.columns and c != 'price']
    corrs = df[cols + ['price']].corr()['price'].drop('price').sort_values(key=abs, ascending=False)
    print(f'\n{city} — correlation with price:')
    print(corrs.round(3).to_string())

---
## OLS Regression — Price Drivers

**Why log(price):** Price is right-skewed. Log transformation makes residuals more symmetric and coefficients interpretable as approximate percentage changes in price.

**Baseline model** (4 features — room type, availability, reviews, minimum nights):
- Singapore R² = 0.432 · Bangkok R² = 0.082

**Enhanced model** — adds three features justified by the data we have:
- `reviews_per_month` — review *rate* (demand velocity), distinct from total review count
- `calculated_host_listings_count` — commercial-scale hosts price differently from individuals
- `log(nb_median_price)` — neighbourhood price context as a continuous signal
- Singapore only: `C(neighbourhood)` fixed effects — ANOVA confirmed neighbourhood is a significant price driver in Singapore (F=4.28, p<0.001); not added for Bangkok where ANOVA p=0.56

**Enhanced model R²:**
- Singapore R² = **0.512** (Adj R² = 0.503) — up from 0.432
- Bangkok R² = **0.178** (Adj R² = 0.178) — up from 0.082

Bangkok's remaining unexplained variance (~82%) reflects unobserved listing quality factors — amenities, interior design, photos — not available in the summary listings file.

In [ ]:
def run_ols(df, city, use_neighbourhood_fe=False):
    cols = ['price', 'room_type', 'neighbourhood', 'availability_365', 'number_of_reviews',
            'minimum_nights', 'reviews_per_month', 'calculated_host_listings_count', 'nb_median_price']
    model_df = df[[c for c in cols if c in df.columns]].dropna().copy()
    model_df['log_price'] = np.log1p(model_df['price'])
    model_df['log_nb_median_price'] = np.log1p(model_df['nb_median_price'])
    model_df['room_type'] = model_df['room_type'].str.replace('/', '_').str.replace(' ', '_')

    # Baseline — 4 features (shown for comparison only)
    base_f = ('log_price ~ C(room_type, Treatment("Private_room"))'
              ' + availability_365 + number_of_reviews + minimum_nights')
    base_r2 = smf.ols(base_f, data=model_df).fit().rsquared

    # Enhanced — adds reviews_per_month, host scale, neighbourhood price context
    # Singapore also gets neighbourhood fixed effects (ANOVA confirmed location matters there)
    nb_fe = ' + C(neighbourhood)' if use_neighbourhood_fe else ''
    enh_f = (f'log_price ~ C(room_type, Treatment("Private_room")){nb_fe}'
             ' + availability_365 + number_of_reviews + minimum_nights'
             ' + reviews_per_month + calculated_host_listings_count + log_nb_median_price')
    model = smf.ols(enh_f, data=model_df).fit()

    print(f'\n{"="*55}')
    print(f'OLS REGRESSION — {city}')
    print(f'{"="*55}')
    print(f'  Baseline R²  : {base_r2:.3f}')
    print(f'  Enhanced R²  : {model.rsquared:.3f}  (+{model.rsquared - base_r2:.3f})')
    print(f'  Adj R²       : {model.rsquared_adj:.3f}')
    print(f'  Observations : {int(model.nobs):,}')
    print(f'  F-statistic  : {model.fvalue:.1f}  (p={model.f_pvalue:.2e})')
    print()

    # Print only non-neighbourhood coefficients to keep output readable
    coef_df = pd.DataFrame({
        'coef':    model.params,
        'p_value': model.pvalues,
        'sig':     model.pvalues.apply(lambda p: '***' if p<0.001 else '**' if p<0.01 else '*' if p<0.05 else '')
    }).round(4)
    mask = ~coef_df.index.str.startswith('C(neighbourhood)')
    print(coef_df[mask].to_string())
    if use_neighbourhood_fe:
        nb_coefs = coef_df[~mask]
        print(f'\n  ... plus {len(nb_coefs)} neighbourhood fixed effects'
              f' (range: {nb_coefs["coef"].min():.3f} to {nb_coefs["coef"].max():.3f})')
    return model

sg_model = run_ols(sg, 'Singapore', use_neighbourhood_fe=True)
bk_model = run_ols(bk, 'Bangkok',   use_neighbourhood_fe=False)

In [ ]:
# Coefficient comparison chart — shared non-neighbourhood terms only
def extract_key_coefs(model):
    params = model.params
    conf   = model.conf_int()
    return pd.DataFrame({
        'coef':  params,
        'lower': conf[0],
        'upper': conf[1],
        'p':     model.pvalues
    })

sg_coefs = extract_key_coefs(sg_model)
bk_coefs = extract_key_coefs(bk_model)

# Keep only terms present in both models and not neighbourhood fixed effects
shared = [
    c for c in sg_coefs.index
    if c in bk_coefs.index
    and c != 'Intercept'
    and not c.startswith('C(neighbourhood)')
]

# Readable axis labels
LABEL_MAP = {
    'availability_365':           'Availability (days)',
    'number_of_reviews':          'Number of reviews',
    'minimum_nights':             'Minimum nights',
    'reviews_per_month':          'Reviews per month',
    'calculated_host_listings_count': 'Host listing count',
    'log_nb_median_price':        'log(Neighbourhood median price)',
}

def clean_label(c):
    if c in LABEL_MAP:
        return LABEL_MAP[c]
    return (c.replace('C(room_type, Treatment("Private_room"))[T.', '')
             .replace(']', '')
             .replace('_', ' '))

fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(shared))
w = 0.35

sg_vals = sg_coefs.loc[shared, 'coef'].values
bk_vals = bk_coefs.loc[shared, 'coef'].values
sg_err  = [(sg_coefs.loc[shared, 'coef'] - sg_coefs.loc[shared, 'lower']).values,
           (sg_coefs.loc[shared, 'upper'] - sg_coefs.loc[shared, 'coef']).values]
bk_err  = [(bk_coefs.loc[shared, 'coef'] - bk_coefs.loc[shared, 'lower']).values,
           (bk_coefs.loc[shared, 'upper'] - bk_coefs.loc[shared, 'coef']).values]

ax.bar(x - w/2, sg_vals, w, yerr=sg_err, label='Singapore (R²=0.512)', color='#2196F3', alpha=0.85, capsize=4)
ax.bar(x + w/2, bk_vals, w, yerr=bk_err, label='Bangkok (R²=0.178)',   color='#FF9800', alpha=0.85, capsize=4)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_xticks(x)
ax.set_xticklabels([clean_label(c) for c in shared], rotation=30, ha='right', fontsize=8)
ax.set_ylabel('Coefficient (log price scale)')
ax.set_title('Enhanced OLS Coefficients: Singapore vs Bangkok\n'
             '(error bars = 95% CI · Singapore includes neighbourhood fixed effects · Bangkok does not)',
             fontsize=11)
ax.legend()
plt.tight_layout()
plt.show()

---
## Cross-City Comparison Summary

Final summary table pulling together all statistical findings.

In [ ]:
summary_rows = []

for df, city, currency, model in [(sg, 'Singapore', 'SGD', sg_model), (bk, 'Bangkok', 'THB', bk_model)]:
    entire  = df[df['room_type'] == 'Entire home/apt']['price'].dropna()
    private = df[df['room_type'] == 'Private room']['price'].dropna()
    _, p_h1 = stats.mannwhitneyu(entire, private, alternative='greater')

    nb_groups = [g['price'].dropna().values for _, g in df.groupby('neighbourhood') if len(g) >= 10]
    _, p_h4   = stats.f_oneway(*nb_groups)

    summary_rows.append({
        'City':                   city,
        'Median price (entire)':  f'{currency} {entire.median():,.0f}',
        'Median price (private)': f'{currency} {private.median():,.0f}',
        'Price premium':          f'{entire.median()/private.median():.2f}x',
        'H1 p-value':             f'{p_h1:.2e}',
        'H4 p-value':             f'{p_h4:.2e}',
        'OLS R² (enhanced)':      f'{model.rsquared:.3f}',
    })

summary = pd.DataFrame(summary_rows).set_index('City')
print('\n=== STATISTICAL FINDINGS SUMMARY ===')
print(summary.T.to_string())

---
## Cross-City Price Comparison in USD

Singapore prices are in SGD, Bangkok prices are in THB. Direct numerical comparison without conversion is misleading — SGD 221 and THB 1,379 appear similar but represent very different real values.

**Conversion applied (Sep 2025 exchange rates):**
- 1 SGD = 0.75 USD
- 1 THB = 0.028 USD

This was done to make cross-city charts interpretable. Local currency columns are preserved for within-city analysis. See Decision D15 in the engineering decision log.

In [ ]:
# Cross-city median price in USD — by room type
usd_comparison = (
    master.groupby(['city', 'room_type'])['price_usd']
    .median()
    .reset_index()
    .rename(columns={'price_usd': 'median_price_usd'})
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Grouped bar: median USD price by room type per city
pivot = usd_comparison.pivot(index='room_type', columns='city', values='median_price_usd').fillna(0)
pivot.plot(kind='bar', ax=axes[0], color=['#FF9800', '#2196F3'], edgecolor='white', width=0.6)
axes[0].set_title('Median Nightly Price by Room Type\n(USD — cross-city comparable)', fontsize=11)
axes[0].set_xlabel('')
axes[0].set_ylabel('Median Price (USD/night)')
axes[0].yaxis.set_major_formatter(mtick.StrMethodFormatter('${x:,.0f}'))
axes[0].tick_params(axis='x', rotation=25)
axes[0].legend(title='City')
for container in axes[0].containers:
    axes[0].bar_label(container, fmt='$%.0f', fontsize=8, padding=2)

# Overall city median comparison
city_usd = master.groupby('city')['price_usd'].median().reset_index()
bars = axes[1].bar(
    city_usd['city'].str.capitalize(),
    city_usd['price_usd'],
    color=['#FF9800', '#2196F3'], edgecolor='white', width=0.4
)
axes[1].set_title('Overall Median Nightly Price\n(USD — cross-city comparable)', fontsize=11)
axes[1].set_ylabel('Median Price (USD/night)')
axes[1].yaxis.set_major_formatter(mtick.StrMethodFormatter('${x:,.0f}'))
for bar, val in zip(bars, city_usd['price_usd']):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                 f'${val:.0f}', ha='center', fontsize=11, fontweight='bold')

plt.suptitle('Singapore vs Bangkok — Price Comparison in USD\n(1 SGD = 0.75 USD  |  1 THB = 0.028 USD  |  Sep 2025 rates)',
             fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

sg_usd = city_usd[city_usd['city'] == 'singapore']['price_usd'].values[0]
bk_usd = city_usd[city_usd['city'] == 'bangkok']['price_usd'].values[0]
print('Median price (USD):')
print(f'  Singapore: ${sg_usd:.2f}')
print(f'  Bangkok:   ${bk_usd:.2f}')
print(f'  Singapore is {sg_usd/bk_usd:.1f}x more expensive than Bangkok in USD terms.')
print()
print('By room type (USD):')
print(pivot.round(2).to_string())

---
## H5 — Weekend vs Weekday Availability

**Background:** Calendar `price` and `adjusted_price` are 100% null — no per-day pricing data was captured in the scrape. However, `available` (t/f) is fully populated for every day.

**Workaround applied in `enrich.py`:** Inner join `calendar.csv.gz` with cleaned listings on `listing_id = id`. This attaches each listing's base price, `room_type`, and `neighbourhood` onto every calendar row. Calendar rows are then tagged with `day_of_week`, `is_weekend` (Sat/Sun = True), and `month`.

**What we can test:** Whether listings are *less available* (more booked out) on weekends vs weekdays. Lower availability on weekends = higher weekend demand. This is a valid proxy for weekend pricing pressure even without per-day rates.

**Test used:** Two-proportions z-test comparing weekend vs weekday availability rates.  
**Null hypothesis (H0):** Weekend and weekday availability rates are identical.  
**Alternative (H5):** Listings are less available (more booked) on weekends.

In [ ]:
---
## Skipped Hypotheses — Data Limitations

| Hypothesis | Reason skipped |
|---|---|
| H2: Superhost vs non-superhost | `host_is_superhost` not present in summary `listings.csv` — requires detailed listings file |
| H3: Review score analysis | `review_scores_rating` not present in summary `listings.csv` — requires detailed listings file |

**Note on H5:** Originally skipped because calendar price columns are 100% null. Resolved by inner-joining calendar with cleaned listings to attach listing base price and availability tags — see H5 section above and `enrich.py` `enrich_calendar()` function.

---
## Skipped Hypotheses — Data Limitations

| Hypothesis | Reason skipped |
|---|---|
| H2: Superhost vs non-superhost | `host_is_superhost` not present in summary `listings.csv` — requires detailed listings file |
| H5: Weekend vs weekday pricing | Both `price` and `adjusted_price` in `calendar.csv.gz` are 100% null — no per-day price data available |

These would be revisited if the detailed `listings.csv` (full metadata version) and a populated calendar file are obtained.